# Pooled Condition 2 Analysis (All Data Through 2026-03-21)

Combines all three condition 2 datasets for the strongest possible test of associative recognition.

| Dataset | n | Study pass | Test split | Design |
|---------|---|-----------|------------|--------|
| Mar 11 | 6 | Single (60 trials) | 24/36 (bug) | old |
| Mar 13 | 6 (c2 only) | Double (120 trials) | 30/30 | new |
| Mar 21 | 26 | Double (120 trials) | 30/30 | new |

**Design differences.** Mar 11 used a single study pass and had an allocation bug producing 24 intact / 36 rearranged test trials (8/12 per emotion). Mar 13 and Mar 21 corrected both issues. The test-phase task (intact vs rearranged judgment) is identical across all three.

### Analyses

**Study phase.** 2(compatibility) x 3(emotion) RM ANOVAs on accuracy and RT.

**Test phase.** 2(pair type) x 3(emotion) RM ANOVAs on p("same") and RT.

**SDT.** d' and criterion per emotion.

**Design comparison.** Old (n=6) vs new (n~31) design comparison on d'.

In [1]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf

# Resolve notebook directory
if '__vsc_ipynb_file__' in dir():
    _nb_dir = Path(__vsc_ipynb_file__).parent
else:
    _candidates = [Path.cwd(), Path.cwd() / 'data']
    _nb_dir = next((p for p in _candidates if (p / '2026_03_21_26subj_c2.csv').exists()), Path.cwd())

# Load all three datasets
df11 = pd.read_csv(_nb_dir / '2026_03_11_pilot_6subj_c2.csv')
df13 = pd.read_csv(_nb_dir / '2026_03_13_pilot_7subj_c2.csv')
df21 = pd.read_csv(_nb_dir / '2026_03_21_26subj_c2.csv')

# Filter Mar 13 to condition 2 only
df13 = df13[df13.condition == 2].copy()

# Tag datasets
df11['pilot'] = 'mar11'
df11['design'] = 'old'
df13['pilot'] = 'mar13'
df13['design'] = 'new'
df21['pilot'] = 'mar21'
df21['design'] = 'new'

# Renumber subjects to be unique across datasets
# Mar 11: 1-6, Mar 13: 7-12, Mar 21: 13-38
n11 = df11.subject_number.nunique()
n13 = df13.subject_number.nunique()

subj_map_13 = {old: new for old, new in zip(
    sorted(df13.subject_number.unique()),
    range(n11 + 1, n11 + n13 + 1)
)}
df13['subject_number'] = df13['subject_number'].map(subj_map_13)

subj_map_21 = {old: new for old, new in zip(
    sorted(df21.subject_number.unique()),
    range(n11 + n13 + 1, n11 + n13 + df21.subject_number.nunique() + 1)
)}
df21['subject_number'] = df21['subject_number'].map(subj_map_21)

# Combine
df = pd.concat([df11, df13, df21], ignore_index=True)
df['correct'] = df['correct'].astype('boolean')

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')
print()

# Verify per pilot
for pilot in ['mar11', 'mar13', 'mar21']:
    pdf = df[df.pilot == pilot]
    n = pdf.subject_number.nunique()
    study_n = pdf[pdf.phase == 'study'].groupby('subject_number').size().unique()
    test_n = pdf[pdf.phase == 'test'].groupby('subject_number').size().unique()
    subjs = sorted(pdf.subject_number.unique())
    print(f'{pilot}: {n} subjects (IDs {subjs[0]}-{subjs[-1]}), '
          f'study trials/subj: {study_n}, test trials/subj: {test_n}')

    # Test split
    test_p = pdf[pdf.phase == 'test']
    if len(test_p) > 0:
        pt = test_p.groupby(['subject_number', 'pair_type']).size().unstack(fill_value=0)
        intact_n = pt['intact'].unique()
        rearr_n = pt['rearranged'].unique()
        print(f'  test split: {intact_n} intact, {rearr_n} rearranged per subject')
    print()

6480 rows, 38 subjects
Conditions: {2: 38}

mar11: 6 subjects (IDs 1-6), study trials/subj: [60], test trials/subj: [60]
  test split: [24] intact, [36] rearranged per subject

mar13: 6 subjects (IDs 7-12), study trials/subj: [120], test trials/subj: [60]
  test split: [30] intact, [30] rearranged per subject

mar21: 26 subjects (IDs 13-38), study trials/subj: [120], test trials/subj: [60]
  test split: [30] intact, [30] rearranged per subject



## Exclusion Criteria

Same criterion as individual analyses: exclude subjects with >=6 zero-correct cells out of 12 in the study-phase design (2 target gender x 2 flanker gender x 3 flanker emotion). Cell sizes differ by design: 5 trials/cell for Mar 11, 10 trials/cell for Mar 13+21.

In [2]:
ZERO_CELL_THRESHOLD = 6

study_all = df[df.phase == 'study']

cell_correct = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).correct.sum().reset_index()
zero_cells = cell_correct.groupby('subject_number').apply(
    lambda g: (g.correct == 0).sum()
).reset_index(name='zero_cells')

subj_study_acc = study_all.groupby('subject_number').correct.mean()
zero_cells['study_accuracy'] = zero_cells.subject_number.map(subj_study_acc)

# Add pilot info
subj_pilot = df.groupby('subject_number').pilot.first()
zero_cells['pilot'] = zero_cells.subject_number.map(subj_pilot)

print('Subjects with zero-correct cells (by pilot):')
has_zeros = zero_cells[zero_cells.zero_cells > 0].sort_values('zero_cells', ascending=False)
if len(has_zeros) == 0:
    print('  None')
else:
    for _, row in has_zeros.iterrows():
        flag = ' ** EXCLUDED' if row.zero_cells >= ZERO_CELL_THRESHOLD else ''
        print(f'  Subject {int(row.subject_number)} ({row.pilot}): '
              f'{int(row.zero_cells)}/12 zero cells, {row.study_accuracy:.1%} accuracy{flag}')
print()

excluded = zero_cells[zero_cells.zero_cells >= ZERO_CELL_THRESHOLD].subject_number.tolist()
keep = zero_cells[zero_cells.zero_cells < ZERO_CELL_THRESHOLD].subject_number.tolist()
df = df[df.subject_number.isin(keep)].copy()

print(f'{len(excluded)} excluded, {df.subject_number.nunique()} subjects remain')
print()
print('Subjects per pilot after exclusion:')
for pilot in ['mar11', 'mar13', 'mar21']:
    n = df[df.pilot == pilot].subject_number.nunique()
    design = 'old' if pilot == 'mar11' else 'new'
    print(f'  {pilot} ({design}): {n}')
print(f'  Total: {df.subject_number.nunique()}')
print(f'  Old design: {df[df.design == "old"].subject_number.nunique()}')
print(f'  New design: {df[df.design == "new"].subject_number.nunique()}')

Subjects with zero-correct cells (by pilot):
  Subject 37 (mar21): 6/12 zero cells, 46.7% accuracy ** EXCLUDED
  Subject 7 (mar13): 4/12 zero cells, 48.3% accuracy
  Subject 11 (mar13): 4/12 zero cells, 50.8% accuracy
  Subject 19 (mar21): 4/12 zero cells, 51.7% accuracy
  Subject 24 (mar21): 4/12 zero cells, 52.5% accuracy
  Subject 38 (mar21): 4/12 zero cells, 50.0% accuracy
  Subject 35 (mar21): 2/12 zero cells, 50.8% accuracy
  Subject 5 (mar11): 1/12 zero cells, 75.0% accuracy
  Subject 36 (mar21): 1/12 zero cells, 60.0% accuracy

1 excluded, 37 subjects remain

Subjects per pilot after exclusion:
  mar11 (old): 6
  mar13 (new): 6
  mar21 (new): 25
  Total: 37
  Old design: 6
  New design: 31


### Exclusion summary

1 subject excluded (Subject 37/mar21: 6/12 zero-correct cells, 46.7% study accuracy). Eight other subjects across pilots had 1-4 zero cells but did not meet the threshold. **37 subjects remain** (6 old design + 31 new design).

## Study Phase (Orienting)

2(flanker gender compatibility) x 3(flanker emotion) RM ANOVAs on accuracy and RT. Note: Mar 11 subjects contributed 60 study trials (5/cell), Mar 13+21 contributed 120 (10/cell). The ANOVA uses subject-level cell means regardless of trial count.

In [3]:
study = df[df.phase == 'study'].copy()
study['compatible'] = study.target_gender == study.flanker_gender
study['compat_label'] = study.compatible.map({True: 'compatible', False: 'incompatible'})

study_acc_subj = study.groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).correct.mean().reset_index(name='accuracy')

study_rt_subj = study[~study.timed_out].groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

acc_table = study_acc_subj.groupby(['compat_label', 'flanker_emotion']).accuracy.agg(
    ['mean', 'std']
).round(3)
print('Study-phase accuracy by compatibility x emotion:')
print(acc_table.to_string())
print()

print('Marginal means (accuracy):')
print(f'  Compatible:   {study_acc_subj[study_acc_subj.compat_label == "compatible"].accuracy.mean():.3f}')
print(f'  Incompatible: {study_acc_subj[study_acc_subj.compat_label == "incompatible"].accuracy.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_acc_subj[study_acc_subj.flanker_emotion == emo].accuracy.mean():.3f}')
print()

rt_table = study_rt_subj.groupby(['compat_label', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print('Study-phase RT (ms) by compatibility x emotion:')
print(rt_table.to_string())
print()

print('Marginal means (RT):')
print(f'  Compatible:   {study_rt_subj[study_rt_subj.compat_label == "compatible"].mean_rt.mean():.1f}')
print(f'  Incompatible: {study_rt_subj[study_rt_subj.compat_label == "incompatible"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_rt_subj[study_rt_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Study-phase accuracy by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry            0.947  0.081
             happy            0.961  0.067
             neutral          0.953  0.067
incompatible angry            0.742  0.346
             happy            0.761  0.352
             neutral          0.751  0.354

Marginal means (accuracy):
  Compatible:   0.954
  Incompatible: 0.751
         angry: 0.845
         happy: 0.861
       neutral: 0.852

Study-phase RT (ms) by compatibility x emotion:
                                mean    std
compat_label flanker_emotion               
compatible   angry             999.5  245.0
             happy             986.5  231.3
             neutral           976.1  256.4
incompatible angry            1117.5  381.1
             happy            1094.1  330.7
             neutral          1122.8  354.9

Marginal means (RT):
  Compatible:   987.4
  Incompatible: 1111.5
 

In [4]:
import math

def _betacf(a, b, x):
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)

def rm_anova_oneway(groups):
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    ms_err = ss_err / df2 if df2 > 0 else float('nan')
    f_val = (ss_cond / df1) / ms_err if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta, ms_err

def rm_anova_twoway(data, a_levels, b_levels):
    a = len(a_levels)
    b = len(b_levels)
    n = len(data[(a_levels[0], b_levels[0])])
    Y = [[[data[(a_levels[j], b_levels[k])][i]
           for k in range(b)] for j in range(a)] for i in range(n)]
    gm = sum(Y[i][j][k] for i in range(n) for j in range(a) for k in range(b)) / (n * a * b)
    subj_m = [sum(Y[i][j][k] for j in range(a) for k in range(b)) / (a * b) for i in range(n)]
    a_m = [sum(Y[i][j][k] for i in range(n) for k in range(b)) / (n * b) for j in range(a)]
    b_m = [sum(Y[i][j][k] for i in range(n) for j in range(a)) / (n * a) for k in range(b)]
    ab_m = [[sum(Y[i][j][k] for i in range(n)) / n for k in range(b)] for j in range(a)]
    sa_m = [[sum(Y[i][j][k] for k in range(b)) / b for j in range(a)] for i in range(n)]
    sb_m = [[sum(Y[i][j][k] for j in range(a)) / a for k in range(b)] for i in range(n)]
    ss_a = n * b * sum((a_m[j] - gm) ** 2 for j in range(a))
    ss_b = n * a * sum((b_m[k] - gm) ** 2 for k in range(b))
    ss_ab = n * sum((ab_m[j][k] - a_m[j] - b_m[k] + gm) ** 2
                    for j in range(a) for k in range(b))
    ss_s = a * b * sum((subj_m[i] - gm) ** 2 for i in range(n))
    ss_as = b * sum((sa_m[i][j] - a_m[j] - subj_m[i] + gm) ** 2
                    for i in range(n) for j in range(a))
    ss_bs = a * sum((sb_m[i][k] - b_m[k] - subj_m[i] + gm) ** 2
                    for i in range(n) for k in range(b))
    ss_total = sum((Y[i][j][k] - gm) ** 2
                   for i in range(n) for j in range(a) for k in range(b))
    ss_abs = ss_total - ss_a - ss_b - ss_ab - ss_s - ss_as - ss_bs
    df_a, df_b, df_ab = a - 1, b - 1, (a - 1) * (b - 1)
    df_as, df_bs, df_abs = df_a * (n - 1), df_b * (n - 1), df_ab * (n - 1)
    results = {}
    for label, ss_eff, df_eff, ss_e, df_e in [
        ('A', ss_a, df_a, ss_as, df_as),
        ('B', ss_b, df_b, ss_bs, df_bs),
        ('AxB', ss_ab, df_ab, ss_abs, df_abs),
    ]:
        ms_eff = ss_eff / df_eff if df_eff > 0 else 0
        ms_e = ss_e / df_e if df_e > 0 else float('nan')
        f_val = ms_eff / ms_e if ms_e > 0 else float('nan')
        p = f_p(f_val, df_eff, df_e)
        eta = ss_eff / (ss_eff + ss_e) if (ss_eff + ss_e) > 0 else 0
        results[label] = {
            'F': f_val, 'df1': df_eff, 'df2': df_e,
            'p': p, 'eta_sq': eta, 'ms_error': ms_e
        }
    return results

def anova_followup(means_a, means_b, ms_error, df_error, label_a, label_b):
    n = len(means_a)
    diff = sum(a - b for a, b in zip(means_a, means_b)) / n
    se = math.sqrt(2 * ms_error / n)
    if se == 0:
        return 0.0, df_error, 1.0, diff
    t_val = diff / se
    p = t_p_twotail(t_val, df_error)
    return t_val, df_error, p, diff

def edge_correct(rate, n):
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate

def indep_t_test(group1, group2):
    """Independent samples t-test (equal variance assumed)."""
    n1, n2 = len(group1), len(group2)
    m1, m2 = sum(group1) / n1, sum(group2) / n2
    ss1 = sum((x - m1) ** 2 for x in group1)
    ss2 = sum((x - m2) ** 2 for x in group2)
    df_val = n1 + n2 - 2
    sp2 = (ss1 + ss2) / df_val
    se = math.sqrt(sp2 * (1/n1 + 1/n2))
    if se == 0:
        return 0.0, df_val, 1.0, m1 - m2
    t_val = (m1 - m2) / se
    p = t_p_twotail(t_val, df_val)
    return t_val, df_val, p, m1 - m2


# --- Study-phase 2x3 ANOVAs ---

subjects = sorted(study_acc_subj.subject_number.unique())
n_subj = len(subjects)
a_levels = ['compatible', 'incompatible']
b_levels = ['angry', 'happy', 'neutral']

acc_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_acc_subj.compat_label == al) & (study_acc_subj.flanker_emotion == bl)
        vals = study_acc_subj[mask].set_index('subject_number').loc[subjects, 'accuracy'].tolist()
        acc_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on accuracy (n={n_subj}):')
print()
acc_results = rm_anova_twoway(acc_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = acc_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_rt_subj.compat_label == al) & (study_rt_subj.flanker_emotion == bl)
        vals = study_rt_subj[mask].set_index('subject_number').loc[subjects, 'mean_rt'].tolist()
        rt_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on RT (n={n_subj}):')
print()
rt_results = rm_anova_twoway(rt_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = rt_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")

2(compatibility) x 3(emotion) RM ANOVA on accuracy (n=37):

  Compatibility: F(1,36) = 12.595, p = 0.001, partial eta^2 = 0.259
  Emotion: F(2,72) = 1.156, p = 0.320, partial eta^2 = 0.031
  Compatibility x Emotion: F(2,72) = 0.047, p = 0.955, partial eta^2 = 0.001

2(compatibility) x 3(emotion) RM ANOVA on RT (n=37):

  Compatibility: F(1,36) = 14.262, p = 0.001, partial eta^2 = 0.284
  Emotion: F(2,72) = 0.754, p = 0.474, partial eta^2 = 0.021
  Compatibility x Emotion: F(2,72) = 0.960, p = 0.388, partial eta^2 = 0.026


### Study phase interpretation

Flanker-compatibility effects are highly significant in the pooled sample. Compatible trials (.95) are more accurate than incompatible (.75), F(1,36) = 12.60, p = .001, partial eta^2 = .26. Compatible trials are also faster (987 ms vs 1112 ms), F(1,36) = 14.26, p = .001, partial eta^2 = .28. No main effect of emotion and no interaction in either measure (all Fs < 1.2, all ps > .32).

The orienting task produces robust, replicable flanker interference across all three datasets.

## Test Phase (Associative Recognition)

2(pair type: intact/rearranged) x 3(flanker emotion) RM ANOVAs on p("same") and RT.

**Cell sizes vary by design.** Mar 11: 8 intact + 12 rearranged per emotion. Mar 13+21: 10 intact + 10 rearranged per emotion. Subject-level means are computed identically regardless.

In [5]:
test = df[df.phase == 'test'].copy()

test['said_same'] = test.apply(
    lambda r: bool(r.correct) if r.pair_type == 'intact' else not bool(r.correct), axis=1
)

psame_subj = test.groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).said_same.mean().reset_index(name='p_same')

rt_test_subj = test[~test.timed_out].groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

print('Test-phase p("same") by pair_type x emotion:')
psame_table = psame_subj.groupby(['pair_type', 'flanker_emotion']).p_same.agg(
    ['mean', 'std']
).round(3)
print(psame_table.to_string())
print()

print('Marginal means p("same"):')
print(f'  Intact:     {psame_subj[psame_subj.pair_type == "intact"].p_same.mean():.3f}')
print(f'  Rearranged: {psame_subj[psame_subj.pair_type == "rearranged"].p_same.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {psame_subj[psame_subj.flanker_emotion == emo].p_same.mean():.3f}')
print()

print('Test-phase RT (ms) by pair_type x emotion:')
rt_table = rt_test_subj.groupby(['pair_type', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print(rt_table.to_string())
print()

print('Marginal means RT (ms):')
print(f'  Intact:     {rt_test_subj[rt_test_subj.pair_type == "intact"].mean_rt.mean():.1f}')
print(f'  Rearranged: {rt_test_subj[rt_test_subj.pair_type == "rearranged"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {rt_test_subj[rt_test_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Test-phase p("same") by pair_type x emotion:
                             mean    std
pair_type  flanker_emotion              
intact     angry            0.522  0.210
           happy            0.499  0.205
           neutral          0.491  0.204
rearranged angry            0.491  0.211
           happy            0.490  0.219
           neutral          0.486  0.179

Marginal means p("same"):
  Intact:     0.504
  Rearranged: 0.489
         angry: 0.507
         happy: 0.494
       neutral: 0.489

Test-phase RT (ms) by pair_type x emotion:
                              mean    std
pair_type  flanker_emotion               
intact     angry            1294.9  349.7
           happy            1285.1  385.7
           neutral          1288.3  382.2
rearranged angry            1266.3  370.6
           happy            1285.3  395.1
           neutral          1258.5  338.3

Marginal means RT (ms):
  Intact:     1289.4
  Rearranged: 1270.1
         angry: 1280.6
         happy: 1285.2
 

In [6]:
test_subjects = sorted(psame_subj.subject_number.unique())
n_test = len(test_subjects)
pt_levels = ['intact', 'rearranged']
emo_levels = ['angry', 'happy', 'neutral']

psame_data = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (psame_subj.pair_type == pt) & (psame_subj.flanker_emotion == emo)
        vals = psame_subj[mask].set_index('subject_number').loc[test_subjects, 'p_same'].tolist()
        psame_data[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on p("same") (n={n_test}):')
print()
psame_results = rm_anova_twoway(psame_data, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = psame_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data_test = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (rt_test_subj.pair_type == pt) & (rt_test_subj.flanker_emotion == emo)
        vals = rt_test_subj[mask].set_index('subject_number').loc[test_subjects, 'mean_rt'].tolist()
        rt_data_test[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on RT (n={n_test}):')
print()
rt_test_results = rm_anova_twoway(rt_data_test, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = rt_test_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

mse_int_psame = psame_results['AxB']['ms_error']
df_int_psame = psame_results['AxB']['df2']
mse_int_rt = rt_test_results['AxB']['ms_error']
df_int_rt = rt_test_results['AxB']['df2']

print('Follow-up comparisons on p("same"):')
print()
print('  Intact vs rearranged within each emotion (discrimination):')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', emo)], psame_data[('rearranged', emo)],
        mse_int_psame, df_int_psame, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within intact (hit rate modulation):')
emo_pairs = [('angry', 'happy'), ('angry', 'neutral'), ('happy', 'neutral')]
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', e1)], psame_data[('intact', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within rearranged (FA rate modulation):')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('rearranged', e1)], psame_data[('rearranged', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('Follow-up comparisons on RT:')
print()
print('  Intact vs rearranged within each emotion:')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', emo)], rt_data_test[('rearranged', emo)],
        mse_int_rt, df_int_rt, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within intact:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', e1)], rt_data_test[('intact', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within rearranged:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('rearranged', e1)], rt_data_test[('rearranged', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')

2(pair_type) x 3(emotion) RM ANOVA on p("same") (n=37):

  Pair type: F(1,36) = 0.523, p = 0.474, partial eta^2 = 0.014
  Emotion: F(2,72) = 0.449, p = 0.640, partial eta^2 = 0.012
  Pair type x Emotion: F(2,72) = 0.243, p = 0.785, partial eta^2 = 0.007

2(pair_type) x 3(emotion) RM ANOVA on RT (n=37):

  Pair type: F(1,36) = 1.214, p = 0.278, partial eta^2 = 0.033
  Emotion: F(2,72) = 0.174, p = 0.840, partial eta^2 = 0.005
  Pair type x Emotion: F(2,72) = 0.319, p = 0.728, partial eta^2 = 0.009

Follow-up comparisons on p("same"):

  Intact vs rearranged within each emotion (discrimination):
    angry: t(72) = 1.105, p = 0.273, diff = 0.030
    happy: t(72) = 0.330, p = 0.743, diff = 0.009
    neutral: t(72) = 0.190, p = 0.850, diff = 0.005

  Pairwise emotion within intact (hit rate modulation):
    angry vs happy: t(72) = 0.841, p = 0.403, diff = 0.023
    angry vs neutral: t(72) = 1.113, p = 0.269, diff = 0.030
    happy vs neutral: t(72) = 0.272, p = 0.786, diff = 0.007

  Pairwi

### Test phase interpretation

**With n=37, associative discrimination remains non-significant.** The pair type main effect on p("same") is F(1,36) = 0.52, p = .474, eta^2 = .014. Subjects responded "same" to 50.4% of intact pairs and 48.9% of rearranged pairs — a 1.5 percentage point difference that is well within noise.

No emotion effects, no interaction (all Fs < 0.5, all ps > .64). RT shows the same null pattern: no pair type effect (F = 1.21, p = .278), no interactions.

The angry condition again shows the largest (but non-significant) discrimination: 3.0 pp difference, t(72) = 1.11, p = .273. Happy and neutral are essentially zero.

This is the definitive test. With 37 subjects, even a small effect (d' ≈ 0.25) should be detectable. The continued null with this sample size strongly argues against a power explanation.

## Supplementary: Signal Detection Analysis

d' and criterion per emotion. Hit = p("same" | intact), FA = p("same" | rearranged). Edge correction: Macmillan & Kaplan (1985).

In [7]:
sdt_per_subj = []
for subj in test_subjects:
    sdata = test[test.subject_number == subj]
    subj_design = sdata.design.iloc[0]
    subj_pilot = sdata.pilot.iloc[0]
    for emotion in emo_levels:
        intact_emo = sdata[(sdata.pair_type == 'intact') & (sdata.flanker_emotion == emotion)]
        rearr_emo = sdata[(sdata.pair_type == 'rearranged') & (sdata.flanker_emotion == emotion)]

        n_intact = len(intact_emo)
        n_rearr = len(rearr_emo)

        hit_rate_raw = intact_emo.said_same.mean() if n_intact > 0 else 0.0
        fa_rate_raw = rearr_emo.said_same.mean() if n_rearr > 0 else 0.0

        hit_rate = edge_correct(hit_rate_raw, n_intact) if n_intact > 0 else 0.5
        fa_rate = edge_correct(fa_rate_raw, n_rearr) if n_rearr > 0 else 0.5

        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))

        sdt_per_subj.append({
            'subject': subj,
            'pilot': subj_pilot,
            'design': subj_design,
            'emotion': emotion,
            'n_intact': n_intact,
            'n_rearranged': n_rearr,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

sdt_df = pd.DataFrame(sdt_per_subj)

sdt_summary = sdt_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    fa_rate_SD=('fa_rate', 'std'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'SDT Analysis (n={n_test} subjects, pooled across all pilots)')
print()
print(sdt_summary.to_string())
print()

dprime_wide = sdt_df.pivot(index='subject', columns='emotion', values='d_prime')
angry_d = dprime_wide['angry'].tolist()
happy_d = dprime_wide['happy'].tolist()
neutral_d = dprime_wide['neutral'].tolist()

f_val_d, df1_d, df2_d, p_d, eta_d, mse_d = rm_anova_oneway([angry_d, happy_d, neutral_d])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, partial eta^2 = {eta_d:.3f}")
print()

print("Follow-up comparisons on d' (using omnibus MSE):")
d_groups = [angry_d, happy_d, neutral_d]
d_labels = ['angry', 'happy', 'neutral']
for i in range(3):
    for j in range(i + 1, 3):
        t, dfe, p, md = anova_followup(d_groups[i], d_groups[j], mse_d, df2_d,
                                        d_labels[i], d_labels[j])
        print(f'  {d_labels[i]} vs {d_labels[j]}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

crit_wide = sdt_df.pivot(index='subject', columns='emotion', values='criterion')
angry_c = crit_wide['angry'].tolist()
happy_c = crit_wide['happy'].tolist()
neutral_c = crit_wide['neutral'].tolist()

f_val_c, df1_c, df2_c, p_c, eta_c, mse_c = rm_anova_oneway([angry_c, happy_c, neutral_c])
print(f"One-way RM ANOVA on criterion (flanker emotion):")
print(f"  F({df1_c},{df2_c}) = {f_val_c:.3f}, p = {p_c:.3f}, partial eta^2 = {eta_c:.3f}")

SDT Analysis (n=37 subjects, pooled across all pilots)

          N  hit_rate_M  hit_rate_SD  fa_rate_M  fa_rate_SD  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                                      
angry    37       0.522        0.210      0.491       0.211      0.076       0.521       -0.031         0.572
happy    37       0.499        0.205      0.490       0.219      0.047       0.616        0.007         0.547
neutral  37       0.491        0.204      0.486       0.179      0.001       0.424        0.030         0.528

One-way RM ANOVA on d' (flanker emotion):
  F(2,72) = 0.231, p = 0.795, partial eta^2 = 0.006

Follow-up comparisons on d' (using omnibus MSE):
  angry vs happy: t(72) = 0.260, p = 0.796, diff = 0.029
  angry vs neutral: t(72) = 0.673, p = 0.503, diff = 0.075
  happy vs neutral: t(72) = 0.414, p = 0.680, diff = 0.046

One-way RM ANOVA on criterion (flanker emotion):
  F(2,72) = 0.623, p = 0

### SDT interpretation

Pooled d' values: angry = 0.08, happy = 0.05, neutral = 0.00. All near zero, no emotion modulation (F < 1, p = .795). Hit rates (.49-.52) and FA rates (.49) hover around .50. Criterion is near zero across emotions (F < 1, p = .539).

The pooled d' values are slightly positive for angry and happy but trivially small — less than a tenth of a standard deviation of the individual d' distributions (SDs .42-.62). This is consistent with sampling noise, not a real signal.

## Design Comparison: Old (Mar 11) vs New (Mar 13+21)

Does doubling the study exposure (old: single pass, new: double pass) change associative discrimination? Independent-samples t-test on d' collapsed across emotion. Note: n=6 for old design, so interpret cautiously.

In [8]:
# Collapse d' across emotions per subject
subj_d = sdt_df.groupby(['subject', 'design']).d_prime.mean().reset_index()

old_d = subj_d[subj_d.design == 'old'].d_prime.tolist()
new_d = subj_d[subj_d.design == 'new'].d_prime.tolist()

print(f'Old design (Mar 11, n={len(old_d)}):')
print(f"  Mean d' = {sum(old_d)/len(old_d):.3f}, SD = {(sum((x - sum(old_d)/len(old_d))**2 for x in old_d)/(len(old_d)-1))**0.5:.3f}")
print(f'New design (Mar 13+21, n={len(new_d)}):')
print(f"  Mean d' = {sum(new_d)/len(new_d):.3f}, SD = {(sum((x - sum(new_d)/len(new_d))**2 for x in new_d)/(len(new_d)-1))**0.5:.3f}")
print()

t, dfe, p, md = indep_t_test(old_d, new_d)
print(f'Independent t-test (old vs new):')
print(f'  t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

# Also show per-pilot breakdown
print('Per-pilot d\' (collapsed across emotion):')
pilot_d = sdt_df.groupby(['subject', 'pilot']).d_prime.mean().reset_index()
for pilot in ['mar11', 'mar13', 'mar21']:
    vals = pilot_d[pilot_d.pilot == pilot].d_prime
    print(f"  {pilot}: M = {vals.mean():.3f}, SD = {vals.std():.3f}, n = {len(vals)}")

Old design (Mar 11, n=6):
  Mean d' = -0.070, SD = 0.389
New design (Mar 13+21, n=31):
  Mean d' = 0.063, SD = 0.348

Independent t-test (old vs new):
  t(35) = -0.845, p = 0.404, diff = -0.133

Per-pilot d' (collapsed across emotion):
  mar11: M = -0.070, SD = 0.389, n = 6
  mar13: M = -0.009, SD = 0.285, n = 6
  mar21: M = 0.081, SD = 0.364, n = 25


### Design comparison interpretation

Doubling the study exposure did not improve associative discrimination. Old design (single pass) d' = -0.07; new design (double pass) d' = 0.06. The difference is not significant, t(35) = -0.85, p = .404.

Per-pilot breakdown shows a slight upward drift (mar11: -0.07, mar13: -0.01, mar21: +0.08) but this is within the range of sampling variability and none of the individual pilot d' values differ significantly from zero.

**The extra study exposure is not the bottleneck.** The failure of associative memory is not about insufficient encoding strength — it is about the incidental encoding task not promoting binding between target and flanker identities.

## Summary

### Practical implications

**The pooled analysis (n=37) is the strongest evidence yet that this paradigm does not produce associative memory.** Key findings:

1. **Robust study-phase effects.** Flanker compatibility is highly significant on both accuracy (p = .001) and RT (p = .001), confirming subjects process the flanker during encoding.

2. **Zero associative discrimination.** Pair type F < 1 on both p("same") and RT. Pooled d' = 0.04 (averaged across emotions). No emotion modulation.

3. **Design doesn't matter.** Single vs double study pass produces no difference in d' (p = .404). The problem is not encoding strength.

4. **The dissociation with condition 1 is clear.** Condition 1 (n=32) shows reliable item recognition (d' ≈ 0.44, p < .001). Condition 2 (n=37) shows no associative recognition. Same incidental encoding task, same stimuli, different test demands — and completely different outcomes.

**Conclusion.** Across 38 subjects (37 analyzed), three data collections, and two design variants, the gender-judgment flanker task does not produce measurable associative binding between target and flanker face identities. Flanker faces are processed (as shown by compatibility effects) but not bound into retrievable pair representations.

In [9]:
n_total = 38
n_analyzed = df.subject_number.nunique()
n_excluded = n_total - n_analyzed

print(f'=== Pooled Condition 2 Summary (Mar 11 + Mar 13 + Mar 21) ===')
print(f'{n_total} subjects collected, {n_excluded} excluded, {n_analyzed} analyzed')
print(f'  Old design (Mar 11): {df[df.design == "old"].subject_number.nunique()}')
print(f'  New design (Mar 13+21): {df[df.design == "new"].subject_number.nunique()}')
print()

print('Study phase (orienting):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
r = acc_results['A']
print(f"  Compatibility: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['B']
print(f"  Emotion:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['AxB']
print(f"  Interaction:   F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
print()

print('Test phase (associative recognition):')
r = psame_results['A']
print(f"  p(\"same\") Pair type:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['B']
print(f"  p(\"same\") Emotion:          F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['AxB']
print(f"  p(\"same\") Interaction:      F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['A']
print(f"  RT Pair type:              F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['B']
print(f"  RT Emotion:                F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['AxB']
print(f"  RT Interaction:            F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
print()

print('Supplementary SDT:')
for emotion, row in sdt_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}, fa={row['fa_rate_M']:.2f}")
print(f"  d' ANOVA: F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, eta^2 = {eta_d:.3f}")
print()

print('Design comparison (d\' collapsed across emotion):')
print(f"  Old (n={len(old_d)}): M = {sum(old_d)/len(old_d):.3f}")
print(f"  New (n={len(new_d)}): M = {sum(new_d)/len(new_d):.3f}")
print(f'  t({dfe}) = {t:.3f}, p = {p:.3f}')

=== Pooled Condition 2 Summary (Mar 11 + Mar 13 + Mar 21) ===
38 subjects collected, 1 excluded, 37 analyzed
  Old design (Mar 11): 6
  New design (Mar 13+21): 31

Study phase (orienting):
  Overall accuracy: 84.7%
  Mean RT: 1045 ms
  Compatibility: F(1,36) = 12.595, p = 0.001
  Emotion:       F(2,72) = 1.156, p = 0.320
  Interaction:   F(2,72) = 0.047, p = 0.955

Test phase (associative recognition):
  p("same") Pair type:       F(1,36) = 0.523, p = 0.474, eta^2 = 0.014
  p("same") Emotion:          F(2,72) = 0.449, p = 0.640, eta^2 = 0.012
  p("same") Interaction:      F(2,72) = 0.243, p = 0.785, eta^2 = 0.007
  RT Pair type:              F(1,36) = 1.214, p = 0.278, eta^2 = 0.033
  RT Emotion:                F(2,72) = 0.174, p = 0.840, eta^2 = 0.005
  RT Interaction:            F(2,72) = 0.319, p = 0.728, eta^2 = 0.009

Supplementary SDT:
    angry: d'=0.08 (SD=0.52), c=-0.03, hit=0.52, fa=0.49
    happy: d'=0.05 (SD=0.62), c=0.01, hit=0.50, fa=0.49
  neutral: d'=0.00 (SD=0.42), c=0